# Setup

### Installation (Kafka)

Reference video: [Guides](https://youtu.be/LX5LKBYHmyU?si=hJaSakdetjpwTCCC)

1. Install Java 17 or higher.
   - Check with:
     ```bash
     java -version
     ```
   - On Windows, install a JDK from AdoptOpenJDK/Temurin or Oracle.

2. Download and extract Kafka.
   - Go to the Apache Kafka downloads page and download a binary release for Scala 2.13.
   - Extract the archive and navigate into the folder:
     ```bash
     tar -xzf kafka_2.13-4.2.0.tgz
     cd kafka_2.13-4.2.0
     ```
     (Replace 4.2.0 with the actual version you downloaded.)

3. Configure and run Kafka.
   - Use the bundled `kafka-storage.bat` and `kafka-server-start.bat` commands on Windows.
   - Later sections of this notebook show the exact commands.

### Installation (MongoDB)

Guidance video: [Guides](https://youtu.be/HT_TUkqK9Wg?si=XwWrt3Brl3fX0a9S)

1. Install MongoDB Community Server.
   - On Windows, use the installer from the MongoDB website.
   - Confirm installation by running:
     ```bash
     mongod --version
     mongosh --version
     ```

2. Start MongoDB.
   - Locally, the default URI is `mongodb://127.0.0.1:27017`.
   - Use one of these commands:
     ```bash
     mongod --dbpath "C:\data\db"
     ```
     or if installed as a service:
     ```bash
     net start MongoDB
     ```

3. (Optional) Install the VS Code MongoDB extension.
   - Search for "MongoDB for VS Code" and install it.
   - This is useful for exploring data, but not required for the pipeline.

4. Verify connectivity with `mongosh`.
   - Connect to the server:
     ```bash
     mongosh
     ```

### Create Python virtual environment

The code in `src/` depends on Spark, Kafka client, MongoDB driver, and data tooling. Create and install these dependencies before running the project.

```bash
# Navigate to the repository root
cd "c:\Users\user\Downloads\FIT3182_ASGN2"

# Create a virtual environment
py -3.12 -m venv .venv

# Activate the environment
.venv\Scripts\activate

# Upgrade pip first
python -m pip install --upgrade pip

# Install required Python modules
python -m pip install pyspark pymongo kafka-python pandas matplotlib
```

> Note: If your local environment already has Python and Spark, you can still create a virtual environment to isolate this project. The `pyspark` package installs a Python-accessible Spark runtime.

### Notes on source structure

- `src/config.py` contains Kafka, MongoDB, camera, and streaming configuration values.
- `src/producer_utils.py` reads CSV files from `data/` and publishes events to Kafka topics.
- `src/streaming_app.py` reads Kafka events, joins camera records, detects violations, and writes results to MongoDB.
- `src/rules.py` contains the violation detection logic used by the streaming application.
- `src/visualisation_utils.py` can read violations from MongoDB and create summary plots.


# Prerequisite (Before impl)

## mongodb

Since the server is actively running in this command prompt window, it is "occupying" this terminal. To run commands to deep clean (drop) your databases, you have two options. You can either **leave this window open** and open a **new** Command Prompt window, or use a graphical tool like MongoDB Compass.

Here is the cleanest way to wipe the data using the command line:

### Step 1: Open a New Command Prompt

Leave your current terminal running (if you close it, MongoDB will shut down). Open a **second** Command Prompt window.

### Step 2: Connect via mongosh

In the new window, type the following command to open the MongoDB Shell:

```cmd
mongosh

```

*(Note: If you are using an older installation, the command might be mongo, but modern versions use mongosh)*.

### Step 3: Deep Clean Your Data

Once you are connected to the shell (you'll see a prompt changing to something like test>), you can clean your data using these commands:

* **To wipe out a specific database completely:**
```javascript
use your_database_name
db.dropDatabase()


```



```
*   **To clean just a single collection (table) inside a database without losing the database itself:**
    ```javascript
    use your_database_name
    db.your_collection_name.drop()
    

```

---

### Alternative: The "Nuclear" Option (Fresh Start)

If you are doing local development and want to delete **absolutely everything** to start completely fresh from scratch:

1. Go back to your original Command Prompt window and press **Ctrl + C** to safely shut down the MongoDB server.
2. Open your Windows File Explorer and navigate to **C:\data\db**.
3. Select everything inside that folder and delete it.
4. Run your original startup command again:
```cmd
mongod --dbpath "C:\data\db"


```
---
## kafka


### What is the kraft-combined-logs folder?

In KRaft mode, Kafka merges its two main responsibilities into a single node setup: acting as a **Broker** (handling your message data) and a **Controller** (managing cluster metadata and coordination). Because it acts as both, it creates a **"combined"** storage directory.

Inside kraft-combined-logs, Kafka stores:

* **Your Data Topics:** Every message, event, partition, and offset you produce to Kafka.
* **Cluster Metadata (__cluster_metadata):** The internal ledger that tracks which brokers exist, what topics exist, and who is in charge.
* **The Cluster ID:** A hidden unique tracking file (meta.properties) that binds your data folder to a specific cluster generation token.

---

### Why do you need to remove it before starting Kafka again?

If you are developing locally and trying to reset Kafka, simply restarting the command script often fails with an error. This happens for two main reasons:

1. **The Cluster ID Mismatch (Most Common Error):**
Every time you run the Kafka setup command to format your storage directory, it generates a brand new **Cluster ID**. If you try to boot Kafka using an *old* kraft-combined-logs folder with a *new* Cluster ID, Kafka will panic and crash because the IDs do not match.
2. **Corrupted Metadata Logs:**
If your previous Kafka instance crashed, was forcefully shut down (Ctrl + C or terminal closure), or has lingering corrupted state records, it will prevent a clean reboot.

> To Kafka, an old data directory looks like a cluster that belongs to someone else. Clearing it allows Kafka to initialize a brand-new, completely clean environment.

---

### How to safely remove and re-apply Kafka

Follow these steps to clean out the logs and start Kafka back up with a clean slate.

1. **Stop Kafka completely:** Required.
Close any active command prompts running Kafka, or press **Ctrl + C** in those windows and wait for the processes to fully shut down.


2. **Delete the log directory:** Deep Clean.
Navigate to your configured Kafka data directory (often found in C:\tmp\kraft-combined-logs or inside your extracted Kafka folder under /data/). Delete the entire **kraft-combined-logs** folder.


3. **Generate a fresh Cluster ID:** Configuration.
Open a command prompt inside your Kafka directory and generate a brand-new cluster UUID string by running:

```cmd
.\bin\windows\kafka-storage.bat random-uuid
.\bin\windows\kafka-storage.bat random-uuid
```

```cmd
    .\bin\windows\kafka-storage.bat format -t YOUR_UUID_HERE -c .\config\kraft\server.properties
    .\bin\windows\kafka-storage.bat format -t YOUR_UUID_HERE -c .\config\kraft\server.properties

```

This recreates a completely fresh, uncorrupted version of the kraft-combined-logs directory.


4. **Launch Kafka:** Ready.
Now you can safely boot Kafka back up with a clean slate:

```cmd
.\bin\windows\kafka-server-start.bat .\config\kraft\server.properties
.\bin\windows\kafka-server-start.bat .\config\kraft\server.properties

```




# Implementation

### Step 0: Activate virtual environment

From the repository root:
```bash
cd "c:\Users\user\Downloads\FIT3182_ASGN2"
.venv\Scripts\activate
```

### Step 1: Start Kafka & MongoDB

Terminal 1 - Start Kafka (using KRaft):
```bash
cd C:\kafka
set KAFKA_HEAP_OPTS=-Xmx1G -Xms1G

# Generate a cluster UUID once per fresh install
.\bin\windows\kafka-storage.bat random-uuid

# Format storage once per fresh install
.\bin\windows\kafka-storage.bat format -t <UUID_FROM_PREVIOUS_COMMAND> -c .\config\server.properties

# Start the Kafka broker
.\bin\windows\kafka-server-start.bat .\config\server.properties
```

Terminal 2 - Start MongoDB:
```bash
mongod --dbpath "C:\data\db"
```

> If MongoDB is installed as a Windows service, use `net start MongoDB` instead.

### Step 2: Create Kafka topics

From the Kafka installation folder in another terminal:
```bash
cd C:\kafka

.\bin\windows\kafka-topics.bat --create --bootstrap-server localhost:9092 --topic camera-events-camera-1 --partitions 1 --replication-factor 1
.\bin\windows\kafka-topics.bat --create --bootstrap-server localhost:9092 --topic camera-events-camera-2 --partitions 1 --replication-factor 1
.\bin\windows\kafka-topics.bat --create --bootstrap-server localhost:9092 --topic camera-events-camera-3 --partitions 1 --replication-factor 1
.\bin\windows\kafka-topics.bat --create --bootstrap-server localhost:9092 --topic speed-violations --partitions 1 --replication-factor 1
```

### Step 3: Run camera producers

Open one terminal per camera producer, activate the same virtual environment, and run from `src/`:
```bash
cd "c:\Users\user\Downloads\FIT3182_ASGN2\src"
python producer_utils.py camera_a
```

```bash
cd "c:\Users\user\Downloads\FIT3182_ASGN2\src"
python producer_utils.py camera_b
```

```bash
cd "c:\Users\user\Downloads\FIT3182_ASGN2\src"
python producer_utils.py camera_c
```

> Each producer publishes events from the corresponding CSV file into the configured Kafka topic.

### Step 4: Run the streaming application

In another terminal, activate the same virtual environment and run from `src/`:
```bash
cd "c:\Users\user\Downloads\FIT3182_ASGN2\src"
python streaming_app.py
```

This application:
- reads events from Kafka topics defined in `src/config.py`
- parses and timestamps each event
- enriches each record with camera speed limits
- performs an event-time join across cameras to detect average-speed violations
- detects instantaneous speed violations
- writes detected violation documents into MongoDB

### Step 5: Verify results in MongoDB

The streaming app writes to the `violations` collection in the `speed_enforcement_db` database.

Use `mongosh` or the MongoDB VS Code Explorer to verify:
```javascript
use speed_enforcement_db
db.violations.find().limit(20).pretty()
```

### Optional: Run visualization

From `src/`:
```bash
cd "c:\Users\user\Downloads\FIT3182_ASGN2\src"
python visualisation_utils.py
```

This script reads the saved violations from MongoDB and creates a summary plot.

### Important notes

- Run Kafka and MongoDB before launching the streaming app.
- If you restart Kafka, you may need to delete the old `kraft-combined-logs` directory and reformat storage.
- `src/producer_utils.py` expects the CSV files in `data/` relative to `src/`.
- `src/streaming_app.py` uses `config.py` values for Kafka bootstrap servers, topic names, and MongoDB settings.
